# MP1 — Prompt Strategy Comparison

This notebook runs four prompting strategies against 10 job snippets and compares their extraction performance (company, role, years_experience_required).
Follow cells in order. Fill `model_call` with your API code before running.

In [ ]:
# Imports
import asyncio
import httpx
import time
import json
from pathlib import Path
from typing import Dict, Any, Optional, List

# Paths
ROOT = Path('.')
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Data loading helpers
SNIPPETS_PATH = DATA_DIR / 'job_snippets.jsonl'
GOLDEN_PATH = DATA_DIR / 'golden_set.jsonl'

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    items = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

snippets = load_jsonl(SNIPPETS_PATH)
golden = {item['id']: item for item in load_jsonl(GOLDEN_PATH)}
len(snippets), len(golden)

In [ ]:
# Prompt variants — each returns a string prompt for one snippet
def prompt_zero_shot(snippet: str) -> str:
    return (
        'Extract the following fields from the job posting.
',
        
        'Job posting:
' + snippet
    )

def prompt_instructional(snippet: str) -> str:
    return (
        'You are an assistant that extracts structured fields from job postings.
',
        
        'Job posting:
' + snippet
    )

def prompt_few_shot(snippet: str) -> str:
    examples = [
        {
            'posting': 'Acme Corp seeks a Senior Engineer with 5+ years of experience.',
            'out': { 'company': 'Acme Corp', 'role': 'Senior Engineer', 'years_experience_required': 5 }
        },
    ]
    ex_text = ''
    for ex in examples:
        ex_text += f